# 🔍 Sensor Anomaly Detection using Graph Neural Networks (GNN)
### Academic Submission: INT423 Advanced Machine Learning / SIH Theme (CA2)
**Author:** Ayush Kumar Tiwari  
**Registration Number:** 12319158  
**Topic:** Non-Euclidean Feature Embedding for Proactive Industrial Resilience using an Adaptive Decoupled Graph Autoencoder

---
## 1. Overview & Mathematical Formulation
This project introduces an unsupervised deep learning framework designed to model complex, interconnected multi-sensor industrial telemetry (e.g. water pumps, turbines).

### Mathematical Framework:
1. **Dynamic Pearson Adjacency:** $A_{i,j} = \mathbb{I}(|\text{Corr}(S_i, S_j)| > \theta)$
2. **Symmetric Laplacian Normalization:** $\tilde{A} = \tilde{D}^{-\frac{1}{2}} (A + I) \tilde{D}^{-\frac{1}{2}}$
3. **Spectral Graph Convolution:** $Z = \text{ReLU}(\tilde{A} X W)$
4. **Reconstruction Residual Anomaly Scoring:** $\text{Score}_t = \frac{1}{N} \sum_{i=1}^N (x_{t,i} - \hat{x}_{t,i})^2$

In [1]:
# Install and import core dependencies
!pip install torch numpy pandas scikit-learn matplotlib
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
print("PyTorch version:", torch.__version__)


## 2. Dynamic Pearson Correlation Graph Induction

In [2]:
def compute_pearson_graph(data, threshold=0.45):
    corr = np.corrcoef(data, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0)
    adj = (np.abs(corr) > threshold).astype(np.float32)
    np.fill_diagonal(adj, 1.0)
    deg = np.sum(adj, axis=1)
    deg_inv_sqrt = np.zeros_like(deg)
    np.power(deg, -0.5, where=deg > 0, out=deg_inv_sqrt)
    d_mat = np.diag(deg_inv_sqrt)
    norm_adj = d_mat @ adj @ d_mat
    return adj, torch.tensor(norm_adj, dtype=torch.float32)


## 3. Graph Autoencoder Architecture

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class GraphAutoencoder(nn.Module):
    def __init__(self, num_sensors, hidden_dim=32, latent_dim=16):
        super().__init__()
        self.encoder_gc1 = nn.Linear(num_sensors, hidden_dim)
        self.encoder_gc2 = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.decoder_fc2 = nn.Linear(hidden_dim, num_sensors)

    def forward(self, x, norm_adj):
        # Spatial graph message passing
        h1 = F.relu(self.encoder_gc1(torch.matmul(x, norm_adj)))
        z = F.relu(self.encoder_gc2(h1))
        h2 = F.relu(self.decoder_fc1(z))
        return self.decoder_fc2(h2)


## 4. End-to-End Pipeline Execution & Evaluation

In [4]:
# Generate synthetic multi-channel pump telemetry
np.random.seed(42)
n_samples, n_sensors = 1200, 12
t = np.linspace(0, 50, n_samples)
signals = np.array([np.sin(t + (i * np.pi / n_sensors)) for i in range(n_sensors)]).T
signals += np.random.normal(0, 0.1, signals.shape)

# Inject anomalous events
labels = np.zeros(n_samples, dtype=int)
anomalies = np.random.choice(n_samples, size=int(n_samples * 0.08), replace=False)
labels[anomalies] = 1
signals[anomalies, 0:3] += np.random.uniform(2.0, 4.0, (len(anomalies), 3))

# Scale and build graph
scaler = MinMaxScaler()
norm_signals = scaler.fit_transform(signals)
adj, norm_adj = compute_pearson_graph(norm_signals, threshold=0.45)
X = torch.tensor(norm_signals, dtype=torch.float32)

# Train Autoencoder
model = GraphAutoencoder(num_sensors=n_sensors)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.MSELoss()

for ep in range(30):
    optimizer.zero_grad()
    recon = model(X, norm_adj)
    loss = loss_fn(recon, X)
    loss.backward()
    optimizer.step()

# Anomaly Scoring via reconstruction residual
model.eval()
with torch.no_grad():
    recon = model(X, norm_adj).numpy()
scores = np.mean((recon - norm_signals) ** 2, axis=1)
threshold = np.percentile(scores, 95)
preds = (scores > threshold).astype(int)

print(f"Accuracy:  {accuracy_score(labels, preds)*100:.2f}%")
print(f"F1 Score:  {f1_score(labels, preds)*100:.2f}%")
print(f"Precision: {precision_score(labels, preds)*100:.2f}%")
print(f"Recall:    {recall_score(labels, preds)*100:.2f}%")
